# Network analysis of learned AKOrN oscillators

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import json
from pathlib import Path
import einops
from einops import rearrange
from sklearn.decomposition import PCA
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Add source directory to path
#sys.path.append('/source')
from source.models.classification.my_knet import MyAKOrN

from source.models.classification.analysis_utils import AKOrNStaticAnalyzer

from source.data.augs import augmentation_strong

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Define the sweep directories
results_dir = Path("results")
sweep_dirs = [
    "sweep_20250708_581384.opbs_0",
    "sweep_20250708_581385.opbs_1",
    "sweep_20250708_581386.opbs_2",
    "sweep_20250708_581387.opbs_3",
    "sweep_20250708_581388.opbs_4",
    "sweep_20250708_581389.opbs_5",
    "sweep_20250708_581390.opbs_6",
    "sweep_20250708_581391.opbs_7",
    "sweep_20250708_581392.opbs_8",
    "sweep_20250708_581393.opbs_9",
    "sweep_20250708_581394.opbs_10",
    "sweep_20250708_581395.opbs_11",
    "sweep_20250708_581396.opbs_12",
    "sweep_20250708_581397.opbs_13",
    "sweep_20250708_581398.opbs_14",
    "sweep_20250708_581399.opbs_15",
    "sweep_20250708_581400.opbs_16",
    "sweep_20250708_581401.opbs_17"
]

print(f"Found {len(sweep_dirs)} sweep directories")

## 1. Load Learned Model and Configuration

In [ ]:
# Load the best model checkpoint
checkpoint_path = "../results/20250704_570979.opbs/my_akorn_cifar10_final.pth"
config_path = "../results/20250704_570979.opbs/parameters.json"

# Load configuration
with open(config_path, 'r') as f:
    config = json.load(f)

print("Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
if 'epoch' in checkpoint_path:
    print(f"\nLoaded checkpoint from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
elif 'final' in checkpoint_path:
    print(f"\nLoaded final checkpoint with accuracy {checkpoint['final_accuracy']:.2f}%")

# Create model with same configuration
model =MyAKOrN(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=config['T'],
    J=config['J'],
    J_bias=config['J_bias'],
    ksizes=config['ksizes'],
    ro_ksize=config['ro_ksize'],
    ro_N=config['ro_N'],
    norm=config['norm'],
    c_norm=config['c_norm'],
    gamma=config['gamma'],
    use_omega=config['use_omega'],
    init_omg=config['init_omg'],
    global_omg=config['global_omg'],
    learn_omg=config['learn_omg'],
    ensemble=config['ensemble']
).to(device)

# Load state dict
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\nModel loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Import network analysis utilities
from source.kuramoto_network_metrics import (
    compute_all_metrics, 
    spectral_metrics, 
    strength_metrics,
    community_metrics,
    path_metrics,
    betweenness_metrics,
    graph_from_K
)
import networkx as nx

print("Network analysis utilities imported successfully!")

In [ ]:
## 2. Extract Connectivity Matrices from Learned Model

def extract_coupling_matrices(model, layer_idx=None):
    """Extract coupling matrices from AKOrN model layers."""
    coupling_matrices = {}
    
    if layer_idx is not None:
        # Extract from specific layer
        layer = model.layers[layer_idx]
        if hasattr(layer[2], 'connectivity'):
            weight = layer[2].connectivity.weight.detach().cpu().numpy()
            coupling_matrices[layer_idx] = weight
            print(f"Layer {layer_idx}: Extracted coupling matrix shape {weight.shape}")
    else:
        # Extract from all layers
        for i, layer in enumerate(model.layers):
            if hasattr(layer[2], 'connectivity'):
                weight = layer[2].connectivity.weight.detach().cpu().numpy()
                coupling_matrices[i] = weight
                print(f"Layer {i}: Extracted coupling matrix shape {weight.shape}")
    
    return coupling_matrices

# Extract coupling matrices from all layers
coupling_matrices = extract_coupling_matrices(model)

print(f"\nExtracted coupling matrices from {len(coupling_matrices)} layers")
for layer_idx, matrix in coupling_matrices.items():
    print(f"  Layer {layer_idx}: {matrix.shape} - range [{matrix.min():.4f}, {matrix.max():.4f}]")

In [ ]:
## 3. Process Connectivity Matrices for Network Analysis

def process_connectivity_for_network_analysis(coupling_matrices):
    """
    Process 4D coupling matrices into 2D coupling matrices suitable for network analysis.
    
    The AKOrN connectivity matrices are typically 4D: (out_channels, in_channels, kernel_h, kernel_w)
    We need to convert them to 2D matrices representing node-to-node coupling.
    """
    processed_matrices = {}
    
    for layer_idx, matrix in coupling_matrices.items():
        print(f"\nProcessing Layer {layer_idx}:")
        print(f"  Original shape: {matrix.shape}")
        
        if matrix.ndim == 4:
            # 4D conv weights: (out_ch, in_ch, h, w)
            out_ch, in_ch, h, w = matrix.shape
            
            # Method 1: Flatten spatial dimensions and reshape
            # This creates a coupling matrix between input and output channels
            matrix_2d = matrix.reshape(out_ch, in_ch * h * w)
            
            # Method 2: Average over spatial dimensions (alternative approach)
            matrix_spatial_avg = matrix.mean(axis=(2, 3))
            
            # Method 3: Frobenius norm over spatial dimensions
            matrix_frob = np.linalg.norm(matrix, axis=(2, 3))
            
            print(f"  Method 1 (flatten spatial): {matrix_2d.shape}")
            print(f"  Method 2 (spatial average): {matrix_spatial_avg.shape}")  
            print(f"  Method 3 (Frobenius norm): {matrix_frob.shape}")
            
            # For network analysis, we'll use the spatial average method
            # and then make it square by taking the correlation matrix if needed
            if matrix_spatial_avg.shape[0] == matrix_spatial_avg.shape[1]:
                # Already square
                processed_matrices[layer_idx] = matrix_spatial_avg
            else:
                # Make square by computing correlation between channels
                if matrix_spatial_avg.shape[0] < matrix_spatial_avg.shape[1]:
                    # More input channels than output channels
                    corr_matrix = np.corrcoef(matrix_spatial_avg)
                    processed_matrices[layer_idx] = corr_matrix
                else:
                    # More output channels than input channels  
                    corr_matrix = np.corrcoef(matrix_spatial_avg.T)
                    processed_matrices[layer_idx] = corr_matrix
                    
        elif matrix.ndim == 2:
            # Already 2D
            processed_matrices[layer_idx] = matrix
            
        else:
            print(f"  Unsupported matrix dimensionality: {matrix.ndim}")
            continue
            
        final_shape = processed_matrices[layer_idx].shape
        print(f"  Final processed shape: {final_shape}")
        print(f"  Is square: {final_shape[0] == final_shape[1]}")
        print(f"  Value range: [{processed_matrices[layer_idx].min():.4f}, {processed_matrices[layer_idx].max():.4f}]")
    
    return processed_matrices

# Process the coupling matrices
processed_coupling_matrices = process_connectivity_for_network_analysis(coupling_matrices)

In [ ]:
## 4. Compute Network Metrics for Each Layer

def analyze_layer_networks(processed_matrices):
    """Compute comprehensive network metrics for each layer."""
    layer_network_metrics = {}
    
    for layer_idx, K in processed_matrices.items():
        print(f"\n=== Network Analysis for Layer {layer_idx} ===")
        print(f"Coupling matrix shape: {K.shape}")
        
        try:
            # Compute all network metrics
            metrics = compute_all_metrics(K, directed=False, threshold=1e-6)
            layer_network_metrics[layer_idx] = metrics
            
            # Print key metrics
            print(f"\\nSpectral metrics:")
            print(f"  λ₂ (algebraic connectivity): {metrics['spectral']['lambda_2']:.6f}")
            print(f"  λ_N (largest eigenvalue): {metrics['spectral']['lambda_N']:.6f}")
            print(f"  Eigenratio (λ_N/λ₂): {metrics['spectral']['eigenratio']:.2f}")
            
            print(f"\\nStrength metrics:")
            strengths = metrics['strength']['strength']
            print(f"  Mean strength: {strengths.mean():.4f}")
            print(f"  Std strength: {strengths.std():.4f}")
            print(f"  Max strength: {strengths.max():.4f}")
            
            print(f"\\nCommunity metrics:")
            print(f"  Modularity Q: {metrics['community']['modularity']:.4f}")
            partition = metrics['community']['partition']
            n_communities = len(set(partition.values()))
            print(f"  Number of communities: {n_communities}")
            
            print(f"\\nPath metrics:")
            print(f"  Average shortest path: {metrics['path']['avg_shortest_path']:.4f}")
            print(f"  Diameter: {metrics['path']['diameter']:.4f}")
            
            # Core metrics
            coreness = metrics['kcore']['coreness']
            print(f"\\nCore metrics:")
            print(f"  Max k-core: {coreness.max()}")
            print(f"  Mean coreness: {coreness.mean():.2f}")
            
        except Exception as e:
            print(f"Error analyzing layer {layer_idx}: {e}")
            layer_network_metrics[layer_idx] = None
    
    return layer_network_metrics

# Analyze all layers
network_metrics = analyze_layer_networks(processed_coupling_matrices)

In [ ]:
## 5. Visualize Network Properties

def plot_network_comparison(network_metrics):
    """Create comprehensive plots comparing network properties across layers."""
    
    # Extract data for plotting
    layers = list(network_metrics.keys())
    valid_layers = [l for l in layers if network_metrics[l] is not None]
    
    if not valid_layers:
        print("No valid network metrics to plot")
        return
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Spectral properties
    lambda2_vals = [network_metrics[l]['spectral']['lambda_2'] for l in valid_layers]
    lambdaN_vals = [network_metrics[l]['spectral']['lambda_N'] for l in valid_layers]
    eigenratios = [network_metrics[l]['spectral']['eigenratio'] for l in valid_layers]
    
    axes[0,0].bar(range(len(valid_layers)), lambda2_vals, alpha=0.7, label='λ₂')
    axes[0,0].bar(range(len(valid_layers)), lambdaN_vals, alpha=0.7, label='λ_N')
    axes[0,0].set_xlabel('Layer')
    axes[0,0].set_ylabel('Eigenvalue')
    axes[0,0].set_title('Laplacian Eigenvalues')
    axes[0,0].set_xticks(range(len(valid_layers)))
    axes[0,0].set_xticklabels([f'L{l}' for l in valid_layers])
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Eigenratio
    axes[0,1].plot(range(len(valid_layers)), eigenratios, 'o-', linewidth=2, markersize=8)
    axes[0,1].set_xlabel('Layer')
    axes[0,1].set_ylabel('Eigenratio (λ_N/λ₂)')
    axes[0,1].set_title('Spectral Gap Ratio')
    axes[0,1].set_xticks(range(len(valid_layers)))
    axes[0,1].set_xticklabels([f'L{l}' for l in valid_layers])
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Strength distribution
    axes[0,2].set_title('Strength Distributions by Layer')
    for i, layer in enumerate(valid_layers):
        strengths = network_metrics[layer]['strength']['strength']
        axes[0,2].hist(strengths, bins=20, alpha=0.6, label=f'Layer {layer}')
    axes[0,2].set_xlabel('Node Strength')
    axes[0,2].set_ylabel('Count')
    axes[0,2].legend()
    axes[0,2].grid(True, alpha=0.3)
    
    # 4. Modularity
    modularities = [network_metrics[l]['community']['modularity'] for l in valid_layers]
    axes[1,0].bar(range(len(valid_layers)), modularities, alpha=0.7, color='green')
    axes[1,0].set_xlabel('Layer')
    axes[1,0].set_ylabel('Modularity Q')
    axes[1,0].set_title('Community Structure')
    axes[1,0].set_xticks(range(len(valid_layers)))
    axes[1,0].set_xticklabels([f'L{l}' for l in valid_layers])
    axes[1,0].grid(True, alpha=0.3)
    
    # 5. Path length metrics
    avg_paths = [network_metrics[l]['path']['avg_shortest_path'] for l in valid_layers]
    diameters = [network_metrics[l]['path']['diameter'] for l in valid_layers]
    
    x = range(len(valid_layers))
    axes[1,1].bar(x, avg_paths, alpha=0.7, label='Avg path length')
    axes[1,1].bar(x, diameters, alpha=0.7, label='Diameter')
    axes[1,1].set_xlabel('Layer')
    axes[1,1].set_ylabel('Path Length')
    axes[1,1].set_title('Path Length Metrics')
    axes[1,1].set_xticks(range(len(valid_layers)))
    axes[1,1].set_xticklabels([f'L{l}' for l in valid_layers])
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    # 6. K-core distribution
    axes[1,2].set_title('K-Core Distributions by Layer')
    for i, layer in enumerate(valid_layers):
        coreness = network_metrics[layer]['kcore']['coreness']
        unique_cores, counts = np.unique(coreness, return_counts=True)
        axes[1,2].bar(unique_cores + i*0.2, counts, width=0.2, alpha=0.7, label=f'Layer {layer}')
    axes[1,2].set_xlabel('K-Core Number')
    axes[1,2].set_ylabel('Count')
    axes[1,2].legend()
    axes[1,2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create the comparison plots
plot_network_comparison(network_metrics)

In [ ]:
## 6. Network Visualization

def visualize_layer_networks(processed_matrices, network_metrics, max_nodes=50):
    """Visualize the actual network graphs for each layer."""
    
    valid_layers = [l for l in network_metrics.keys() if network_metrics[l] is not None]
    
    fig, axes = plt.subplots(1, len(valid_layers), figsize=(6*len(valid_layers), 6))
    if len(valid_layers) == 1:
        axes = [axes]
    
    for i, layer_idx in enumerate(valid_layers):
        K = processed_matrices[layer_idx]
        
        # Limit size for visualization
        if K.shape[0] > max_nodes:
            print(f"Layer {layer_idx}: Reducing from {K.shape[0]} to {max_nodes} nodes for visualization")
            # Take the top nodes by strength
            strengths = network_metrics[layer_idx]['strength']['strength']
            top_indices = np.argsort(strengths)[-max_nodes:]
            K_viz = K[np.ix_(top_indices, top_indices)]
        else:
            K_viz = K
        
        # Create NetworkX graph
        G = graph_from_K(K_viz, directed=False, threshold=np.percentile(np.abs(K_viz), 75))
        
        # Get community partition for coloring
        if K_viz.shape[0] <= max_nodes:
            try:
                community_data = community_metrics(K_viz, directed=False)
                partition = community_data['partition']
                colors = [partition.get(node, 0) for node in G.nodes()]
            except:
                colors = 'blue'
        else:
            colors = 'blue'
        
        # Layout
        if G.number_of_nodes() > 0:
            pos = nx.spring_layout(G, k=1/np.sqrt(G.number_of_nodes()), iterations=50)
            
            # Draw network
            nx.draw(G, pos, ax=axes[i], 
                   node_color=colors, 
                   node_size=50,
                   edge_color='gray',
                   alpha=0.7,
                   with_labels=False)
        
        axes[i].set_title(f'Layer {layer_idx} Network\\n({K_viz.shape[0]} nodes, {G.number_of_edges()} edges)')
    
    plt.tight_layout()
    plt.show()

# Visualize networks
visualize_layer_networks(processed_coupling_matrices, network_metrics)

In [ ]:
## 7. Comparative Analysis Across Sweep Models

def analyze_sweep_models(sweep_dirs, layer_idx=0):
    """Compare network metrics across different models from the parameter sweep."""
    
    sweep_network_data = {}
    
    print(f"Analyzing network properties across {len(sweep_dirs)} sweep models for layer {layer_idx}...")
    
    for sweep_dir in sweep_dirs:
        print(f"\\nProcessing {sweep_dir}...")
        
        # Load configuration
        config_path = results_dir / sweep_dir / "parameters.json"
        model_path = results_dir / sweep_dir / "my_akorn_cifar10_final.pth"
        
        if not config_path.exists() or not model_path.exists():
            print(f"  Skipping {sweep_dir}: missing files")
            continue
            
        try:
            # Load config
            with open(config_path, 'r') as f:
                config = json.load(f)
            
            # Create and load model
            sweep_model = MyAKOrN(
                n=config['n'],
                ch=config['ch'], 
                out_classes=config['num_classes'],
                L=config['L'],
                T=config['T'],
                J=config['J'],
                J_bias=config['J_bias'],
                ksizes=config['ksizes'],
                ro_ksize=config['ro_ksize'],
                ro_N=config['ro_N'],
                norm=config['norm'],
                c_norm=config['c_norm'],
                gamma=config['gamma'],
                use_omega=config['use_omega'],
                init_omg=config['init_omg'],
                global_omg=config['global_omg'],
                learn_omg=config['learn_omg'],
                ensemble=config['ensemble']
            ).to(device)
            
            checkpoint = torch.load(model_path, map_location=device)
            sweep_model.load_state_dict(checkpoint['model_state_dict'])
            sweep_model.eval()
            
            # Extract coupling matrix for specific layer
            coupling_matrices = extract_coupling_matrices(sweep_model, layer_idx=layer_idx)
            if layer_idx not in coupling_matrices:
                print(f"  Layer {layer_idx} not found in {sweep_dir}")
                continue
                
            # Process for network analysis
            processed_matrices = process_connectivity_for_network_analysis(coupling_matrices)
            if layer_idx not in processed_matrices:
                print(f"  Could not process layer {layer_idx} in {sweep_dir}")
                continue
            
            # Compute network metrics
            K = processed_matrices[layer_idx]
            metrics = compute_all_metrics(K, directed=False, threshold=1e-6)
            
            # Store results
            sweep_network_data[sweep_dir] = {
                'config': config,
                'metrics': metrics,
                'gamma': config['gamma'],
                'T': config['T']
            }
            
            print(f"  ✓ Analyzed {sweep_dir}: γ={config['gamma']}, T={config['T']}")
            
        except Exception as e:
            print(f"  ✗ Error with {sweep_dir}: {e}")
            continue
    
    print(f"\\nSuccessfully analyzed {len(sweep_network_data)} models")
    return sweep_network_data

# Analyze sweep models (this might take a while)
sweep_network_data = analyze_sweep_models(sweep_dirs[:5], layer_idx=0)  # Start with first 5 models

In [ ]:
## 8. Analyze Parameter Dependencies

def plot_parameter_dependencies(sweep_network_data):
    """Plot how network metrics depend on gamma and T parameters."""
    
    if len(sweep_network_data) < 2:
        print("Need at least 2 models for parameter dependency analysis")
        return
    
    # Extract data
    gammas = [data['gamma'] for data in sweep_network_data.values()]
    Ts = [data['T'] for data in sweep_network_data.values()]
    
    # Extract network metrics
    lambda2s = [data['metrics']['spectral']['lambda_2'] for data in sweep_network_data.values()]
    eigenratios = [data['metrics']['spectral']['eigenratio'] for data in sweep_network_data.values()]
    modularities = [data['metrics']['community']['modularity'] for data in sweep_network_data.values()]
    avg_paths = [data['metrics']['path']['avg_shortest_path'] for data in sweep_network_data.values()]
    
    # Create plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Eigenratio vs gamma
    axes[0,0].scatter(gammas, eigenratios, c=Ts, cmap='viridis', s=100, alpha=0.7)
    axes[0,0].set_xlabel('Gamma')
    axes[0,0].set_ylabel('Eigenratio (λ_N/λ₂)')
    axes[0,0].set_title('Eigenratio vs Gamma (colored by T)')
    axes[0,0].set_xscale('log')
    cbar1 = plt.colorbar(axes[0,0].collections[0], ax=axes[0,0])
    cbar1.set_label('T value')
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Modularity vs gamma
    axes[0,1].scatter(gammas, modularities, c=Ts, cmap='viridis', s=100, alpha=0.7)
    axes[0,1].set_xlabel('Gamma')
    axes[0,1].set_ylabel('Modularity Q')
    axes[0,1].set_title('Modularity vs Gamma (colored by T)')
    axes[0,1].set_xscale('log')
    cbar2 = plt.colorbar(axes[0,1].collections[0], ax=axes[0,1])
    cbar2.set_label('T value')
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Path length vs T
    axes[1,0].scatter(Ts, avg_paths, c=gammas, cmap='plasma', s=100, alpha=0.7)
    axes[1,0].set_xlabel('T')
    axes[1,0].set_ylabel('Average Path Length')
    axes[1,0].set_title('Path Length vs T (colored by γ)')
    cbar3 = plt.colorbar(axes[1,0].collections[0], ax=axes[1,0])
    cbar3.set_label('Gamma value')
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Algebraic connectivity vs gamma
    axes[1,1].scatter(gammas, lambda2s, c=Ts, cmap='viridis', s=100, alpha=0.7)
    axes[1,1].set_xlabel('Gamma')
    axes[1,1].set_ylabel('λ₂ (Algebraic Connectivity)')
    axes[1,1].set_title('Algebraic Connectivity vs Gamma (colored by T)')
    axes[1,1].set_xscale('log')
    cbar4 = plt.colorbar(axes[1,1].collections[0], ax=axes[1,1])
    cbar4.set_label('T value')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print correlation analysis
    print("\\n=== Parameter Correlation Analysis ===")
    
    # Convert to log scale for gamma
    log_gammas = np.log10(gammas)
    
    correlations = {
        'log(gamma) vs eigenratio': np.corrcoef(log_gammas, eigenratios)[0,1],
        'log(gamma) vs modularity': np.corrcoef(log_gammas, modularities)[0,1],
        'log(gamma) vs λ₂': np.corrcoef(log_gammas, lambda2s)[0,1],
        'T vs avg_path': np.corrcoef(Ts, avg_paths)[0,1],
        'T vs eigenratio': np.corrcoef(Ts, eigenratios)[0,1],
    }
    
    for metric, corr in correlations.items():
        print(f"{metric}: {corr:.3f}")

# Plot parameter dependencies
plot_parameter_dependencies(sweep_network_data)

In [ ]:
## 9. Summary and Interpretation

def generate_network_analysis_summary(network_metrics, sweep_network_data):
    """Generate a comprehensive summary of network analysis findings."""
    
    print("="*60)
    print("NETWORK ANALYSIS SUMMARY")
    print("="*60)
    
    print("\\n1. SINGLE MODEL LAYER ANALYSIS:")
    print("-" * 40)
    
    for layer_idx, metrics in network_metrics.items():
        if metrics is None:
            continue
            
        print(f"\\nLayer {layer_idx}:")
        print(f"  • Spectral properties:")
        print(f"    - λ₂ (algebraic connectivity): {metrics['spectral']['lambda_2']:.6f}")
        print(f"    - Eigenratio (λ_N/λ₂): {metrics['spectral']['eigenratio']:.2f}")
        
        print(f"  • Community structure:")
        print(f"    - Modularity Q: {metrics['community']['modularity']:.4f}")
        n_communities = len(set(metrics['community']['partition'].values()))
        print(f"    - Number of communities: {n_communities}")
        
        print(f"  • Path metrics:")
        print(f"    - Average path length: {metrics['path']['avg_shortest_path']:.4f}")
        print(f"    - Network diameter: {metrics['path']['diameter']:.1f}")
        
        strengths = metrics['strength']['strength']
        print(f"  • Strength distribution:")
        print(f"    - Mean: {strengths.mean():.4f}, Std: {strengths.std():.4f}")
        
        cores = metrics['kcore']['coreness']
        print(f"  • K-core structure:")
        print(f"    - Max k-core: {cores.max()}, Mean: {cores.mean():.2f}")
    
    if len(sweep_network_data) > 1:
        print("\\n\\n2. PARAMETER SWEEP ANALYSIS:")
        print("-" * 40)
        
        # Extract parameter ranges
        gammas = [data['gamma'] for data in sweep_network_data.values()]
        Ts = [data['T'] for data in sweep_network_data.values()]
        eigenratios = [data['metrics']['spectral']['eigenratio'] for data in sweep_network_data.values()]
        modularities = [data['metrics']['community']['modularity'] for data in sweep_network_data.values()]
        
        print(f"\\nParameter ranges analyzed:")
        print(f"  • Gamma: {min(gammas):.3f} - {max(gammas):.3f}")
        print(f"  • T: {min(Ts)} - {max(Ts)}")
        
        print(f"\\nNetwork metric ranges:")
        print(f"  • Eigenratio: {min(eigenratios):.2f} - {max(eigenratios):.2f}")
        print(f"  • Modularity: {min(modularities):.4f} - {max(modularities):.4f}")
        
        # Key findings
        print(f"\\nKey findings:")
        if len(set(gammas)) > 1:
            print(f"  • Gamma variation shows network structure dependency")
        if len(set(Ts)) > 1:
            print(f"  • T parameter affects temporal integration and connectivity")
        
    print("\\n\\n3. KURAMOTO NETWORK INTERPRETATION:")
    print("-" * 40)
    
    # Provide interpretation in context of Kuramoto oscillator networks
    for layer_idx, metrics in network_metrics.items():
        if metrics is None:
            continue
            
        lambda2 = metrics['spectral']['lambda_2']
        eigenratio = metrics['spectral']['eigenratio']
        modularity = metrics['community']['modularity']
        
        print(f"\\nLayer {layer_idx} - Kuramoto dynamics implications:")
        
        if lambda2 > 0.01:
            print(f"  • High algebraic connectivity (λ₂={lambda2:.4f}) → Strong synchronization potential")
        else:
            print(f"  • Low algebraic connectivity (λ₂={lambda2:.6f}) → Weak synchronization")
            
        if eigenratio < 10:
            print(f"  • Low eigenratio ({eigenratio:.2f}) → Good synchronization stability")
        else:
            print(f"  • High eigenratio ({eigenratio:.2f}) → Potential synchronization challenges")
            
        if modularity > 0.3:
            print(f"  • High modularity ({modularity:.3f}) → Clustered synchronization likely")
        else:
            print(f"  • Low modularity ({modularity:.3f}) → Uniform synchronization pattern")
    
    print("\\n" + "="*60)

# Generate comprehensive summary
generate_network_analysis_summary(network_metrics, sweep_network_data)

In [ ]:
# Network Analysis of AKOrN Decomposition Variables
# This notebook analyzes the network properties of key variables from analysis_utils.py

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import torch
from source.models.classification.analysis_utils import AKOrNStaticAnalyzer
from source.models.classification.my_knet import MyAKOrN
import networkx as nx
from sklearn.metrics import pairwise_distances
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, spearmanr
import pandas as pd

print("Network analysis setup complete!")

In [ ]:
## Network Analysis of AKOrN Decomposition Variables

def extract_analysis_variables(model, layer_idx=0):
    """Extract the target variables from a trained AKOrN model layer."""
    
    # Initialize analyzer
    analyzer = AKOrNStaticAnalyzer(model, layer_idx)
    
    # Extract connectivity weights and blocks
    connectivity_data = analyzer.extract_connectivity_weights()
    if connectivity_data is None:
        print(f"Could not extract connectivity from layer {layer_idx}")
        return None
    
    # Extract 2x2 blocks
    blocks = analyzer.extract_connectivity_blocks()
    if blocks is None:
        print(f"Could not extract connectivity blocks from layer {layer_idx}")
        return None
    
    print(f"Layer {layer_idx}: Extracted {blocks.shape[0]} connectivity blocks")
    
    # Compute frob_norms
    frob_norms = np.linalg.norm(blocks, axis=(1, 2))
    
    # Compute symmetric/skew decomposition
    try:
        p1, p2, p3, q, sym_frob, skew_frob = analyzer.decompose_symmetric_skew(blocks)
    except Exception as e:
        print(f"Error in symmetric/skew decomposition: {e}")
        return None
    
    # Compute rotation/symmetric decomposition
    try:
        c_R, c_S, alpha, beta = analyzer.decompose_rotation_symmetric(blocks)
    except Exception as e:
        print(f"Error in rotation/symmetric decomposition: {e}")
        return None
    
    # Package results
    variables = {
        'frob_norms': frob_norms,
        'sym_frob': sym_frob,
        'skew_frob': skew_frob,
        'c_R': c_R,
        'c_S': c_S,
        'connectivity_blocks': blocks,
        'num_blocks': blocks.shape[0],
        'layer_idx': layer_idx
    }
    
    print(f"Successfully extracted all variables:")
    for var_name, var_data in variables.items():
        if isinstance(var_data, np.ndarray):
            print(f"  {var_name}: shape={var_data.shape}, range=[{var_data.min():.4f}, {var_data.max():.4f}]")
        else:
            print(f"  {var_name}: {var_data}")
    
    return variables

# Extract variables from the loaded model
analysis_vars = extract_analysis_variables(model, layer_idx=0)

if analysis_vars is not None:
    print("\n✓ Successfully extracted analysis variables for network analysis")
else:
    print("\n✗ Failed to extract analysis variables")

In [ ]:
## 1. Correlation Network Analysis

def create_correlation_networks(variables):
    """Create correlation networks from the analysis variables."""
    
    # Target variables for network analysis
    target_vars = ['frob_norms', 'sym_frob', 'skew_frob', 'c_R', 'c_S']
    
    # Create data matrix
    data_matrix = np.column_stack([variables[var] for var in target_vars])
    var_names = target_vars
    
    print(f"Data matrix shape: {data_matrix.shape}")
    print(f"Variables: {var_names}")
    
    # Compute correlation matrices
    pearson_corr = np.corrcoef(data_matrix.T)
    
    # Compute Spearman correlation
    spearman_corr = np.zeros((len(var_names), len(var_names)))
    for i in range(len(var_names)):
        for j in range(len(var_names)):
            spearman_corr[i, j] = spearmanr(data_matrix[:, i], data_matrix[:, j])[0]
    
    # Create distance matrices (1 - |correlation|)
    pearson_dist = 1 - np.abs(pearson_corr)
    spearman_dist = 1 - np.abs(spearman_corr)
    
    # Create NetworkX graphs
    def create_graph_from_correlation(corr_matrix, var_names, threshold=0.5):
        G = nx.Graph()
        G.add_nodes_from(var_names)
        
        for i in range(len(var_names)):
            for j in range(i+1, len(var_names)):
                if abs(corr_matrix[i, j]) >= threshold:
                    G.add_edge(var_names[i], var_names[j], 
                             weight=abs(corr_matrix[i, j]),
                             correlation=corr_matrix[i, j])
        return G
    
    # Create correlation graphs
    pearson_graph = create_graph_from_correlation(pearson_corr, var_names, threshold=0.3)
    spearman_graph = create_graph_from_correlation(spearman_corr, var_names, threshold=0.3)
    
    results = {
        'data_matrix': data_matrix,
        'var_names': var_names,
        'pearson_corr': pearson_corr,
        'spearman_corr': spearman_corr,
        'pearson_dist': pearson_dist,
        'spearman_dist': spearman_dist,
        'pearson_graph': pearson_graph,
        'spearman_graph': spearman_graph
    }
    
    return results

# Create correlation networks
if analysis_vars is not None:
    corr_networks = create_correlation_networks(analysis_vars)
    print("✓ Created correlation networks")
    
    # Print network statistics
    print(f"\nPearson correlation network:")
    print(f"  Nodes: {corr_networks['pearson_graph'].number_of_nodes()}")
    print(f"  Edges: {corr_networks['pearson_graph'].number_of_edges()}")
    
    print(f"\nSpearman correlation network:")
    print(f"  Nodes: {corr_networks['spearman_graph'].number_of_nodes()}")
    print(f"  Edges: {corr_networks['spearman_graph'].number_of_edges()}")
else:
    print("✗ Cannot create correlation networks - no analysis variables")

In [ ]:
## 2. Similarity Network Analysis

def create_similarity_networks(variables):
    """Create similarity networks based on connectivity blocks."""
    
    blocks = variables['connectivity_blocks']  # Shape: (num_blocks, 2, 2)
    target_vars = ['frob_norms', 'sym_frob', 'skew_frob', 'c_R', 'c_S']
    
    # Create feature matrix for each block
    feature_matrix = np.column_stack([variables[var] for var in target_vars])
    
    print(f"Feature matrix shape: {feature_matrix.shape}")
    print(f"Number of connectivity blocks: {blocks.shape[0]}")
    
    # Compute pairwise distances/similarities
    # Using a subset for computational efficiency
    max_blocks = min(1000, blocks.shape[0])
    indices = np.random.choice(blocks.shape[0], max_blocks, replace=False)
    
    feature_subset = feature_matrix[indices]
    blocks_subset = blocks[indices]
    
    print(f"Using subset of {max_blocks} blocks for similarity analysis")
    
    # Compute different distance metrics
    euclidean_dist = pairwise_distances(feature_subset, metric='euclidean')
    cosine_dist = pairwise_distances(feature_subset, metric='cosine')
    manhattan_dist = pairwise_distances(feature_subset, metric='manhattan')
    
    # Convert distances to similarities
    euclidean_sim = 1 / (1 + euclidean_dist)
    cosine_sim = 1 - cosine_dist
    manhattan_sim = 1 / (1 + manhattan_dist)
    
    # Create NetworkX graphs from similarity matrices
    def create_similarity_graph(sim_matrix, threshold=0.8):
        G = nx.Graph()
        n = sim_matrix.shape[0]
        G.add_nodes_from(range(n))
        
        for i in range(n):
            for j in range(i+1, n):
                if sim_matrix[i, j] >= threshold:
                    G.add_edge(i, j, weight=sim_matrix[i, j])
        return G
    
    # Create similarity graphs with different thresholds
    euclidean_graph = create_similarity_graph(euclidean_sim, threshold=0.9)
    cosine_graph = create_similarity_graph(cosine_sim, threshold=0.8)
    manhattan_graph = create_similarity_graph(manhattan_sim, threshold=0.9)
    
    results = {
        'feature_matrix': feature_subset,
        'blocks_subset': blocks_subset,
        'indices': indices,
        'euclidean_dist': euclidean_dist,
        'cosine_dist': cosine_dist,
        'manhattan_dist': manhattan_dist,
        'euclidean_sim': euclidean_sim,
        'cosine_sim': cosine_sim,
        'manhattan_sim': manhattan_sim,
        'euclidean_graph': euclidean_graph,
        'cosine_graph': cosine_graph,
        'manhattan_graph': manhattan_graph
    }
    
    return results

# Create similarity networks
if analysis_vars is not None:
    sim_networks = create_similarity_networks(analysis_vars)
    print("✓ Created similarity networks")
    
    # Print network statistics
    print(f"\nEuclidean similarity network:")
    print(f"  Nodes: {sim_networks['euclidean_graph'].number_of_nodes()}")
    print(f"  Edges: {sim_networks['euclidean_graph'].number_of_edges()}")
    
    print(f"\nCosine similarity network:")
    print(f"  Nodes: {sim_networks['cosine_graph'].number_of_nodes()}")
    print(f"  Edges: {sim_networks['cosine_graph'].number_of_edges()}")
    
    print(f"\nManhattan similarity network:")
    print(f"  Nodes: {sim_networks['manhattan_graph'].number_of_nodes()}")
    print(f"  Edges: {sim_networks['manhattan_graph'].number_of_edges()}")
else:
    print("✗ Cannot create similarity networks - no analysis variables")

In [ ]:
## 3. Network Visualization and Analysis

def visualize_correlation_networks(corr_networks):
    """Visualize the correlation networks."""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Pearson correlation heatmap
    im1 = axes[0, 0].imshow(corr_networks['pearson_corr'], cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0, 0].set_title('Pearson Correlation Matrix', fontsize=16)
    axes[0, 0].set_xticks(range(len(corr_networks['var_names'])))
    axes[0, 0].set_yticks(range(len(corr_networks['var_names'])))
    axes[0, 0].set_xticklabels(corr_networks['var_names'], rotation=45, fontsize=13)
    axes[0, 0].set_yticklabels(corr_networks['var_names'], fontsize=13)
    
    # Add correlation values as text
    for i in range(len(corr_networks['var_names'])):
        for j in range(len(corr_networks['var_names'])):
            text = axes[0, 0].text(j, i, f'{corr_networks["pearson_corr"][i, j]:.3f}',
                                 ha="center", va="center", color="black", fontsize=11)
    
    plt.colorbar(im1, ax=axes[0, 0])
    
    # 2. Spearman correlation heatmap
    im2 = axes[0, 1].imshow(corr_networks['spearman_corr'], cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0, 1].set_title('Spearman Correlation Matrix', fontsize=16)
    axes[0, 1].set_xticks(range(len(corr_networks['var_names'])))
    axes[0, 1].set_yticks(range(len(corr_networks['var_names'])))
    axes[0, 1].set_xticklabels(corr_networks['var_names'], rotation=45, fontsize=13)
    axes[0, 1].set_yticklabels(corr_networks['var_names'], fontsize=13)
    
    # Add correlation values as text
    for i in range(len(corr_networks['var_names'])):
        for j in range(len(corr_networks['var_names'])):
            text = axes[0, 1].text(j, i, f'{corr_networks["spearman_corr"][i, j]:.3f}',
                                 ha="center", va="center", color="black", fontsize=11)
    
    plt.colorbar(im2, ax=axes[0, 1])
    
    # 3. Pearson correlation network
    G_pearson = corr_networks['pearson_graph']
    if G_pearson.number_of_edges() > 0:
        pos = nx.spring_layout(G_pearson, k=2, iterations=50)
        edge_weights = [G_pearson[u][v]['weight'] for u, v in G_pearson.edges()]
        edge_colors = [G_pearson[u][v]['correlation'] for u, v in G_pearson.edges()]
        
        nx.draw_networkx_nodes(G_pearson, pos, ax=axes[1, 0], node_color='lightblue', 
                              node_size=2000, alpha=0.8)
        edges = nx.draw_networkx_edges(G_pearson, pos, ax=axes[1, 0], 
                                      width=[w*5 for w in edge_weights],
                                      edge_color=edge_colors, edge_cmap=plt.cm.RdBu_r,
                                      edge_vmin=-1, edge_vmax=1)
        nx.draw_networkx_labels(G_pearson, pos, ax=axes[1, 0], font_size=10)
        axes[1, 0].set_title('Pearson Correlation Network', fontsize=16)
        axes[1, 0].axis('off')
    else:
        axes[1, 0].text(0.5, 0.5, 'No edges in Pearson network\\n(threshold too high)', 
                       ha='center', va='center', transform=axes[1, 0].transAxes, fontsize=14)
        axes[1, 0].set_title('Pearson Correlation Network', fontsize=16)
    
    # 4. Spearman correlation network
    G_spearman = corr_networks['spearman_graph']
    if G_spearman.number_of_edges() > 0:
        pos = nx.spring_layout(G_spearman, k=2, iterations=50)
        edge_weights = [G_spearman[u][v]['weight'] for u, v in G_spearman.edges()]
        edge_colors = [G_spearman[u][v]['correlation'] for u, v in G_spearman.edges()]
        
        nx.draw_networkx_nodes(G_spearman, pos, ax=axes[1, 1], node_color='lightcoral', 
                              node_size=2000, alpha=0.8)
        edges = nx.draw_networkx_edges(G_spearman, pos, ax=axes[1, 1], 
                                      width=[w*5 for w in edge_weights],
                                      edge_color=edge_colors, edge_cmap=plt.cm.RdBu_r,
                                      edge_vmin=-1, edge_vmax=1)
        nx.draw_networkx_labels(G_spearman, pos, ax=axes[1, 1], font_size=10)
        axes[1, 1].set_title('Spearman Correlation Network', fontsize=16)
        axes[1, 1].axis('off')
    else:
        axes[1, 1].text(0.5, 0.5, 'No edges in Spearman network\\n(threshold too high)', 
                       ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=14)
        axes[1, 1].set_title('Spearman Correlation Network', fontsize=16)
    
    plt.tight_layout()
    plt.show()

# Visualize correlation networks
if analysis_vars is not None and 'corr_networks' in locals():
    visualize_correlation_networks(corr_networks)
else:
    print("✗ Cannot visualize correlation networks - no data available")

In [ ]:
## 4. Clustering Analysis of Connectivity Blocks

def analyze_connectivity_clusters(variables, n_clusters=5):
    """Perform clustering analysis on connectivity blocks using the decomposition variables."""
    
    target_vars = ['frob_norms', 'sym_frob', 'skew_frob', 'c_R', 'c_S']
    
    # Create feature matrix
    feature_matrix = np.column_stack([variables[var] for var in target_vars])
    
    # Standardize features
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    feature_matrix_scaled = scaler.fit_transform(feature_matrix)
    
    print(f"Feature matrix shape: {feature_matrix_scaled.shape}")
    
    # Perform K-means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(feature_matrix_scaled)
    
    # Compute silhouette score
    from sklearn.metrics import silhouette_score
    silhouette_avg = silhouette_score(feature_matrix_scaled, cluster_labels)
    
    print(f"Number of clusters: {n_clusters}")
    print(f"Silhouette score: {silhouette_avg:.3f}")
    
    # Analyze cluster characteristics
    cluster_stats = {}
    for i in range(n_clusters):
        cluster_mask = cluster_labels == i
        cluster_size = np.sum(cluster_mask)
        cluster_features = feature_matrix[cluster_mask]
        
        cluster_stats[i] = {
            'size': cluster_size,
            'percentage': cluster_size / len(cluster_labels) * 100,
            'mean_features': cluster_features.mean(axis=0),
            'std_features': cluster_features.std(axis=0)
        }
        
        print(f"\\nCluster {i}: {cluster_size} blocks ({cluster_size/len(cluster_labels)*100:.1f}%)")
        for j, var_name in enumerate(target_vars):
            print(f"  {var_name}: {cluster_features.mean(axis=0)[j]:.4f} ± {cluster_features.std(axis=0)[j]:.4f}")
    
    # Create cluster network
    # Connect blocks that are in the same cluster
    cluster_graph = nx.Graph()
    
    # Add nodes (limited subset for visualization)
    max_nodes = min(500, len(cluster_labels))
    subset_indices = np.random.choice(len(cluster_labels), max_nodes, replace=False)
    subset_labels = cluster_labels[subset_indices]
    
    cluster_graph.add_nodes_from(range(max_nodes))
    
    # Add edges between nodes in the same cluster
    for i in range(max_nodes):
        for j in range(i+1, max_nodes):
            if subset_labels[i] == subset_labels[j]:
                cluster_graph.add_edge(i, j, cluster=subset_labels[i])
    
    print(f"\\nCluster network:")
    print(f"  Nodes: {cluster_graph.number_of_nodes()}")
    print(f"  Edges: {cluster_graph.number_of_edges()}")
    
    results = {
        'feature_matrix': feature_matrix,
        'feature_matrix_scaled': feature_matrix_scaled,
        'cluster_labels': cluster_labels,
        'cluster_stats': cluster_stats,
        'silhouette_score': silhouette_avg,
        'kmeans_model': kmeans,
        'cluster_graph': cluster_graph,
        'subset_indices': subset_indices,
        'subset_labels': subset_labels,
        'n_clusters': n_clusters
    }
    
    return results

# Perform clustering analysis
if analysis_vars is not None:
    clustering_results = analyze_connectivity_clusters(analysis_vars, n_clusters=5)
    print("\\n✓ Completed clustering analysis")
else:
    print("✗ Cannot perform clustering analysis - no analysis variables")

In [ ]:
## 5. Comprehensive Network Analysis Visualization

def visualize_network_analyses(corr_networks, sim_networks, clustering_results):
    """Create comprehensive visualization of all network analyses."""
    
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Variable distributions
    ax1 = plt.subplot(3, 4, 1)
    target_vars = ['frob_norms', 'sym_frob', 'skew_frob', 'c_R', 'c_S']
    data_matrix = corr_networks['data_matrix']
    
    for i, var_name in enumerate(target_vars):
        ax1.hist(data_matrix[:, i], bins=50, alpha=0.6, label=var_name)
    ax1.set_title('Variable Distributions', fontsize=16)
    ax1.set_xlabel('Value', fontsize=13)
    ax1.set_ylabel('Frequency', fontsize=13)
    ax1.legend(fontsize=11)
    ax1.tick_params(axis='both', which='major', labelsize=13)
    
    # 2. Correlation matrix
    ax2 = plt.subplot(3, 4, 2)
    im = ax2.imshow(corr_networks['pearson_corr'], cmap='RdBu_r', vmin=-1, vmax=1)
    ax2.set_title('Pearson Correlation', fontsize=16)
    ax2.set_xticks(range(len(target_vars)))
    ax2.set_yticks(range(len(target_vars)))
    ax2.set_xticklabels(target_vars, rotation=45, fontsize=11)
    ax2.set_yticklabels(target_vars, fontsize=11)
    plt.colorbar(im, ax=ax2, shrink=0.8)
    
    # 3. Similarity network (cosine)
    ax3 = plt.subplot(3, 4, 3)
    G_sim = sim_networks['cosine_graph']
    if G_sim.number_of_edges() > 0:
        pos = nx.spring_layout(G_sim, k=0.5, iterations=50)
        nx.draw_networkx_nodes(G_sim, pos, ax=ax3, node_color='lightgreen', 
                              node_size=30, alpha=0.8)
        nx.draw_networkx_edges(G_sim, pos, ax=ax3, alpha=0.3, width=0.5)
        ax3.set_title(f'Cosine Similarity Network\\n({G_sim.number_of_nodes()} nodes, {G_sim.number_of_edges()} edges)', fontsize=14)
    else:
        ax3.text(0.5, 0.5, 'No edges\\n(threshold too high)', 
                ha='center', va='center', transform=ax3.transAxes, fontsize=12)
        ax3.set_title('Cosine Similarity Network', fontsize=14)
    ax3.axis('off')
    
    # 4. Cluster network
    ax4 = plt.subplot(3, 4, 4)
    G_cluster = clustering_results['cluster_graph']
    if G_cluster.number_of_edges() > 0:
        pos = nx.spring_layout(G_cluster, k=0.3, iterations=50)
        node_colors = [clustering_results['subset_labels'][i] for i in G_cluster.nodes()]
        nx.draw_networkx_nodes(G_cluster, pos, ax=ax4, node_color=node_colors, 
                              node_size=20, alpha=0.8, cmap='tab10')
        nx.draw_networkx_edges(G_cluster, pos, ax=ax4, alpha=0.2, width=0.3)
        ax4.set_title(f'Cluster Network\\n({G_cluster.number_of_nodes()} nodes, {G_cluster.number_of_edges()} edges)', fontsize=14)
    else:
        ax4.text(0.5, 0.5, 'No cluster edges', 
                ha='center', va='center', transform=ax4.transAxes, fontsize=12)
        ax4.set_title('Cluster Network', fontsize=14)
    ax4.axis('off')
    
    # 5. Pairwise variable relationships
    ax5 = plt.subplot(3, 4, 5)
    ax5.scatter(data_matrix[:, 0], data_matrix[:, 1], alpha=0.5, s=1)
    ax5.set_xlabel('frob_norms', fontsize=13)
    ax5.set_ylabel('sym_frob', fontsize=13)
    ax5.set_title('Frob Norms vs Sym Frob', fontsize=16)
    ax5.tick_params(axis='both', which='major', labelsize=13)
    
    ax6 = plt.subplot(3, 4, 6)
    ax6.scatter(data_matrix[:, 3], data_matrix[:, 4], alpha=0.5, s=1)
    ax6.set_xlabel('c_R', fontsize=13)
    ax6.set_ylabel('c_S', fontsize=13)
    ax6.set_title('c_R vs c_S', fontsize=16)
    ax6.tick_params(axis='both', which='major', labelsize=13)
    
    ax7 = plt.subplot(3, 4, 7)
    ax7.scatter(data_matrix[:, 1], data_matrix[:, 2], alpha=0.5, s=1)
    ax7.set_xlabel('sym_frob', fontsize=13)
    ax7.set_ylabel('skew_frob', fontsize=13)
    ax7.set_title('Sym Frob vs Skew Frob', fontsize=16)
    ax7.tick_params(axis='both', which='major', labelsize=13)
    
    # 8. Cluster characteristics
    ax8 = plt.subplot(3, 4, 8)
    cluster_sizes = [clustering_results['cluster_stats'][i]['size'] for i in range(clustering_results['n_clusters'])]
    colors = plt.cm.tab10(np.linspace(0, 1, clustering_results['n_clusters']))
    ax8.pie(cluster_sizes, labels=[f'Cluster {i}' for i in range(clustering_results['n_clusters'])], 
           autopct='%1.1f%%', colors=colors)
    ax8.set_title('Cluster Size Distribution', fontsize=16)
    
    # 9-12. Distance matrices
    ax9 = plt.subplot(3, 4, 9)
    im9 = ax9.imshow(sim_networks['euclidean_dist'][:100, :100], cmap='viridis')
    ax9.set_title('Euclidean Distance (100x100)', fontsize=14)
    ax9.set_xticks([])
    ax9.set_yticks([])
    plt.colorbar(im9, ax=ax9, shrink=0.8)
    
    ax10 = plt.subplot(3, 4, 10)
    im10 = ax10.imshow(sim_networks['cosine_dist'][:100, :100], cmap='viridis')
    ax10.set_title('Cosine Distance (100x100)', fontsize=14)
    ax10.set_xticks([])
    ax10.set_yticks([])
    plt.colorbar(im10, ax=ax10, shrink=0.8)
    
    ax11 = plt.subplot(3, 4, 11)
    im11 = ax11.imshow(sim_networks['manhattan_dist'][:100, :100], cmap='viridis')
    ax11.set_title('Manhattan Distance (100x100)', fontsize=14)
    ax11.set_xticks([])
    ax11.set_yticks([])
    plt.colorbar(im11, ax=ax11, shrink=0.8)
    
    # 12. Cluster feature means
    ax12 = plt.subplot(3, 4, 12)
    cluster_means = np.array([clustering_results['cluster_stats'][i]['mean_features'] 
                             for i in range(clustering_results['n_clusters'])])
    im12 = ax12.imshow(cluster_means, cmap='RdBu_r', aspect='auto')
    ax12.set_title('Cluster Feature Means', fontsize=16)
    ax12.set_xticks(range(len(target_vars)))
    ax12.set_yticks(range(clustering_results['n_clusters']))
    ax12.set_xticklabels(target_vars, rotation=45, fontsize=11)
    ax12.set_yticklabels([f'C{i}' for i in range(clustering_results['n_clusters'])], fontsize=11)
    plt.colorbar(im12, ax=ax12, shrink=0.8)
    
    plt.tight_layout()
    plt.show()

# Create comprehensive visualization
if (analysis_vars is not None and 'corr_networks' in locals() and 
    'sim_networks' in locals() and 'clustering_results' in locals()):
    visualize_network_analyses(corr_networks, sim_networks, clustering_results)
else:
    print("✗ Cannot create comprehensive visualization - missing data")

In [ ]:
## 6. Statistical Analysis and Summary

def generate_network_analysis_summary(analysis_vars, corr_networks, sim_networks, clustering_results):
    """Generate comprehensive statistical summary of network analysis results."""
    
    print("="*80)
    print("NETWORK ANALYSIS SUMMARY: AKOrN DECOMPOSITION VARIABLES")
    print("="*80)
    
    # Basic statistics
    print("\\n1. DATA OVERVIEW:")
    print("-" * 40)
    print(f"Layer analyzed: {analysis_vars['layer_idx']}")
    print(f"Total connectivity blocks: {analysis_vars['num_blocks']:,}")
    print(f"Variables analyzed: frob_norms, sym_frob, skew_frob, c_R, c_S")
    
    target_vars = ['frob_norms', 'sym_frob', 'skew_frob', 'c_R', 'c_S']
    data_matrix = corr_networks['data_matrix']
    
    print(f"\\nVariable statistics:")
    for i, var_name in enumerate(target_vars):
        var_data = data_matrix[:, i]
        print(f"  {var_name:12s}: mean={var_data.mean():.4f}, std={var_data.std():.4f}, "
              f"range=[{var_data.min():.4f}, {var_data.max():.4f}]")
    
    # Correlation analysis
    print("\\n\\n2. CORRELATION ANALYSIS:")
    print("-" * 40)
    pearson_corr = corr_networks['pearson_corr']
    spearman_corr = corr_networks['spearman_corr']
    
    print("Pearson correlation matrix:")
    print("Variables:", target_vars)
    for i, var1 in enumerate(target_vars):
        correlation_str = f"  {var1:12s}: "
        for j, var2 in enumerate(target_vars):
            if i != j:
                correlation_str += f"{var2}={pearson_corr[i,j]:+.3f} "
        print(correlation_str)
    
    # Find strongest correlations
    print("\\nStrongest correlations (|r| > 0.5):")
    for i in range(len(target_vars)):
        for j in range(i+1, len(target_vars)):
            if abs(pearson_corr[i, j]) > 0.5:
                print(f"  {target_vars[i]} ↔ {target_vars[j]}: r={pearson_corr[i,j]:+.3f}")
    
    # Clustering analysis
    print("\\n\\n3. CLUSTERING ANALYSIS:")
    print("-" * 40)
    print(f"Number of clusters: {clustering_results['n_clusters']}")
    print(f"Silhouette score: {clustering_results['silhouette_score']:.3f}")
    
    print("\\nCluster characteristics:")
    for i in range(clustering_results['n_clusters']):
        stats = clustering_results['cluster_stats'][i]
        print(f"  Cluster {i}: {stats['size']:,} blocks ({stats['percentage']:.1f}%)")
        
        # Find dominant characteristics
        mean_features = stats['mean_features']
        feature_ranks = np.argsort(mean_features)[::-1]  # Descending order
        print(f"    Dominant features: {target_vars[feature_ranks[0]]}({mean_features[feature_ranks[0]]:.3f}), "
              f"{target_vars[feature_ranks[1]]}({mean_features[feature_ranks[1]]:.3f})")
    
    # Network topology analysis
    print("\\n\\n4. NETWORK TOPOLOGY:")
    print("-" * 40)
    
    # Correlation networks
    G_pearson = corr_networks['pearson_graph']
    G_spearman = corr_networks['spearman_graph']
    
    print("Correlation networks:")
    print(f"  Pearson network: {G_pearson.number_of_nodes()} nodes, {G_pearson.number_of_edges()} edges")
    print(f"  Spearman network: {G_spearman.number_of_nodes()} nodes, {G_spearman.number_of_edges()} edges")
    
    # Similarity networks
    G_euclidean = sim_networks['euclidean_graph']
    G_cosine = sim_networks['cosine_graph']
    G_manhattan = sim_networks['manhattan_graph']
    
    print("\\nSimilarity networks:")
    print(f"  Euclidean network: {G_euclidean.number_of_nodes()} nodes, {G_euclidean.number_of_edges()} edges")
    print(f"  Cosine network: {G_cosine.number_of_nodes()} nodes, {G_cosine.number_of_edges()} edges")
    print(f"  Manhattan network: {G_manhattan.number_of_nodes()} nodes, {G_manhattan.number_of_edges()} edges")
    
    # Cluster network
    G_cluster = clustering_results['cluster_graph']
    print(f"\\nCluster network: {G_cluster.number_of_nodes()} nodes, {G_cluster.number_of_edges()} edges")
    
    # Interpretation
    print("\\n\\n5. INTERPRETATION:")
    print("-" * 40)
    
    # Analyze relationships between decomposition variables
    frob_sym_corr = pearson_corr[0, 1]  # frob_norms vs sym_frob
    sym_skew_corr = pearson_corr[1, 2]  # sym_frob vs skew_frob
    cr_cs_corr = pearson_corr[3, 4]     # c_R vs c_S
    
    print("Key findings:")
    
    if abs(frob_sym_corr) > 0.8:
        print(f"  • Strong correlation between total magnitude and symmetric component (r={frob_sym_corr:.3f})")
        print("    → Connectivity blocks are dominated by symmetric structures")
    elif abs(frob_sym_corr) > 0.5:
        print(f"  • Moderate correlation between total magnitude and symmetric component (r={frob_sym_corr:.3f})")
    
    if abs(sym_skew_corr) > 0.3:
        print(f"  • Correlation between symmetric and skew components (r={sym_skew_corr:.3f})")
        print("    → Symmetric and skew structures are not independent")
    else:
        print(f"  • Weak correlation between symmetric and skew components (r={sym_skew_corr:.3f})")
        print("    → Symmetric and skew structures are relatively independent")
    
    if abs(cr_cs_corr) > 0.5:
        print(f"  • Strong correlation between rotation and symmetric magnitudes (r={cr_cs_corr:.3f})")
        print("    → Rotation and symmetric components are coupled")
    else:
        print(f"  • Weak correlation between rotation and symmetric magnitudes (r={cr_cs_corr:.3f})")
        print("    → Rotation and symmetric components are relatively independent")
    
    # Cluster interpretation
    largest_cluster = max(clustering_results['cluster_stats'].items(), key=lambda x: x[1]['size'])
    cluster_id, cluster_data = largest_cluster
    print(f"\\n  • Largest cluster ({cluster_id}) contains {cluster_data['percentage']:.1f}% of blocks")
    
    dominant_feature_idx = np.argmax(cluster_data['mean_features'])
    dominant_feature = target_vars[dominant_feature_idx]
    print(f"    → Characterized by high {dominant_feature} values")
    
    if clustering_results['silhouette_score'] > 0.5:
        print(f"  • Good cluster separation (silhouette={clustering_results['silhouette_score']:.3f})")
        print("    → Connectivity blocks form distinct groups")
    else:
        print(f"  • Moderate cluster separation (silhouette={clustering_results['silhouette_score']:.3f})")
        print("    → Connectivity blocks have gradual transitions")
    
    print("\\n" + "="*80)

# Generate comprehensive summary
if (analysis_vars is not None and 'corr_networks' in locals() and 
    'sim_networks' in locals() and 'clustering_results' in locals()):
    generate_network_analysis_summary(analysis_vars, corr_networks, sim_networks, clustering_results)
else:
    print("✗ Cannot generate summary - missing analysis data")